# Phase 2 — Build & Train the Model
AI Foundations Bootcamp | MNIST Capstone

> **AI assistance used for code structure — cited per academic honesty policy.**

## 0. Imports & Seeds

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import pandas as pd
import os

# Fix seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

os.makedirs('../models', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

print('TensorFlow version:', tf.__version__)

## 1. Load & Preprocess Data

In [ ]:
# Load MNIST
(X_full, y_full), (X_test, y_test) = keras.datasets.mnist.load_data()

# Scale pixels to [0, 1]
X_full = X_full / 255.0
X_test  = X_test  / 255.0

# Add channel dimension: (28,28) -> (28,28,1)
X_full = X_full[..., np.newaxis]
X_test  = X_test[...,  np.newaxis]

# Split: 6000 validation, rest for training
X_val,   y_val   = X_full[:6000],  y_full[:6000]
X_train, y_train = X_full[6000:],  y_full[6000:]

print(f'Train : {X_train.shape}  |  Val : {X_val.shape}  |  Test : {X_test.shape}')
print('Test set is LOCKED — will not be touched until Phase 3.')

## 2. Visualise Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i, :, :, 0], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/sample_images.png', dpi=100)
plt.show()

## 3. Build the Baseline CNN

In [ ]:
def build_model(dropout_rate=0.0):
    """Build a small CNN. dropout_rate=0 means no dropout (baseline)."""
    model = keras.Sequential([
        # Block 1
        layers.Conv2D(32, 3, activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D(),
        # Block 2
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(),
        # Classifier head
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
    ])
    if dropout_rate > 0:
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(10, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

baseline = build_model()
baseline.summary()

## 4. Experiment Table

Change ONE thing per run and record the result.

In [ ]:
experiment_log = []

def run_experiment(name, dropout=0.0, epochs=5, augment=False):
    """Train one model variant and log the results."""
    np.random.seed(42)
    tf.random.set_seed(42)

    model = build_model(dropout_rate=dropout)

    if augment:
        # Data augmentation (bonus task)
        data_gen = keras.preprocessing.image.ImageDataGenerator(
            rotation_range=10,
            width_shift_range=0.1,
            height_shift_range=0.1
        )
        train_gen = data_gen.flow(X_train, y_train, batch_size=32)
        history = model.fit(
            train_gen,
            validation_data=(X_val, y_val),
            epochs=epochs, verbose=0
        )
    else:
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs, batch_size=32, verbose=0
        )

    best_val = max(history.history['val_accuracy'])
    experiment_log.append({
        'Run': name,
        'Dropout': dropout,
        'Epochs': epochs,
        'Augment': augment,
        'Best Val Accuracy': round(best_val, 4)
    })
    print(f'[{name}]  Best val accuracy: {best_val:.4f}')
    return model, history

In [ ]:
# Run 1 — Baseline
model_baseline, hist_baseline = run_experiment('Run 1 — Baseline', dropout=0.0, epochs=5)

In [ ]:
# Run 2 — Add Dropout
model_dropout, hist_dropout = run_experiment('Run 2 — Dropout 0.25', dropout=0.25, epochs=5)

In [ ]:
# Run 3 — More epochs
model_long, hist_long = run_experiment('Run 3 — Dropout 0.25, 10 epochs', dropout=0.25, epochs=10)

In [ ]:
# Run 4 — Data augmentation (BONUS +2)
model_aug, hist_aug = run_experiment('Run 4 — Augmentation', dropout=0.25, epochs=10, augment=True)

In [ ]:
# Print experiment table
df_log = pd.DataFrame(experiment_log)
print('\n=== Experiment Table ===')
print(df_log.to_string(index=False))

## 5. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for hist, label in [
    (hist_baseline, 'Baseline'),
    (hist_dropout,  'Dropout 0.25'),
    (hist_long,     '10 epochs'),
    (hist_aug,      'Augmentation'),
]:
    axes[0].plot(hist.history['val_accuracy'], label=label)
    axes[1].plot(hist.history['val_loss'],     label=label)

axes[0].set_title('Validation Accuracy per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].set_title('Validation Loss per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/training_curves.png', dpi=100)
plt.show()
print('Saved: reports/training_curves.png')

## 6. Save Best Model

In [ ]:
# Best model = Run 3 (Dropout + 10 epochs) or Run 4 (augmentation)
# Choose the one with highest val accuracy from the experiment table
best_run = df_log.loc[df_log['Best Val Accuracy'].idxmax()]
print('Best run:', best_run['Run'])

# Save the best model
model_long.save('../models/mnist_cnn.h5')
print('Model saved to models/mnist_cnn.h5')